# PV / ASOS CSV 전처리 및 실제 측정시각 정합 — v2

**목적:** 고정식 발전소 A/B CSV, 시간별 ASOS CSV의 컬럼 품질과 수집 간격을 검사하고, **발전소의 약 5분 간격 원본 시각은 그대로 유지**하면서 ASOS 연속형 기상 특징을 해당 시각에 정합합니다. 이미 존재하는 `누적 발전량 차분`과 `출력 비율`은 **재계산하지 않고 검증**합니다.

Python 3.10.20 / pandas / NumPy. PyTorch·가상 발전소 생성·AI 학습·OPC UA는 이 노트북의 범위 밖입니다.

**주의:** ASOS 시간 집계형(강수량·일사량·일조시간), 풍향, 품질 플래그는 자동 선형 보간하거나 시각당 단일 값으로 임의 매핑하지 않습니다. 시간별 결측 보완은 먼저 진행하고, 이어서 양쪽 관측값이 존재하는 구간에서만 인버터의 **실제 시각**에 기상값을 보간합니다. 이는 **과거 데이터 재생 오프라인 분석 전용**으로, 미래 시각 관측값을 활용할 수 있습니다.

In [1]:
import os, re, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
print('Python:',sys.version.split()[0],'pandas:',pd.__version__)


Python: 3.10.20 pandas: 2.3.3


## 0. 파일·단위·정합 정책

`timestamp_col=None`이면 자동 탐색. **파생 특징 컬럼명은 실제 CSV를 확인해 직접 지정**하십시오. 기본값 `None`이면 해당 검사는 건너뜁니다. `energy_unit`은 누적 발전량의 원래 저장 단위이며, `energy_delta_unit`은 **기존 차분 열**의 단위입니다. 해당 열이 없는 예제 CSV도 나머지 프로파일링은 실행됩니다.

- 정격출력과 순간출력은 W로 가정하되 단위 확인 필요
- 누적 발전량과 차분이 동일 인버터의 값인지 검증; 개별 인버터 식별자가 있는 경우 설정 권장
- 약 5분 주기는 **관측 간격 분포로 진단**하며 강제 리샘플링·반올림하지 않음
- 보간 허용되는 연속형 ASOS 컬럼은 화이트리스트로 제한, **최대 2개의 연속 누락 시간 관측**에 한해서만 내부 보간
- 시간별 관측을 실제 약 5분 관측값으로 주장하지 않음. 보간 여부·관측 시각·실제 관측값까지의 거리 기록


In [2]:
DATA_DIR = Path(os.getenv('PV_DATA_DIR','../assets/'))
FILES = {
 'plant_A': {'path': str(DATA_DIR/'inverterdata/inverterdata_62_-261120.csv'), 'timestamp_col':None},
 'plant_B': {'path': str(DATA_DIR/'inverterdata/inverterdata_64_-261120.csv'), 'timestamp_col':None},
 'asos': {'path': str(DATA_DIR/'asosdata/asosdata.csv'), 'timestamp_col':None},
}
OUT=Path(os.getenv('PV_PROFILE_OUT','./profile_output_v2')); OUT.mkdir(parents=True,exist_ok=True)
START=pd.Timestamp('2024-01-01 00:00:00')
END=pd.Timestamp('2026-11-20 23:00:00')
NOMINAL_INTERVAL_MIN=5
GAP_ALERT_MIN=10                  # 주기 이상 구간 보고 임계값 (이 값으로 데이터 삭제하지 않음)
MAX_MISSING_ASOS_HOURS=2         # ASOS 시간별 결측 연속 최대 2개
MAX_ASOS_BRACKET_HOURS=3         # 실측 양 끝 관측시각 거리 상한 (2개 결측이면 3시간)

# 실데이터의 컬럼명을 확인하여 필요시 각 발전소별로 지정
FEATURE_COLUMNS={
 'plant_A': {'device_id':None,'power_w':None,'rated_power_w':None,
             'accum_energy':None,'energy_unit':'Wh', 'energy_delta_wh':None,
             'energy_delta_unit':'Wh','power_ratio':None},
 'plant_B': {'device_id':None,'power_w':None,'rated_power_w':None,
             'accum_energy':None,'energy_unit':'Wh', 'energy_delta_wh':None,
             'energy_delta_unit':'Wh','power_ratio':None},
}
# 온도, 상대습도, 이슬점온도, 현지/해면기압: 실제 CSV 속성 확인 후 확장 가능
WEATHER_ALLOW=[r'^기온(?:\(|$)', r'^상대습도(?:\(|$)',r'^이슬점온도(?:\(|$)',
               r'^현지기압(?:\(|$)',r'^해면기압(?:\(|$)',r'^기압(?:\(|$)']
WEATHER_BLOCK=[r'강수',r'일사',r'일조',r'풍향',r'풍속',r'운량',r'구름',r'적설',r'품질',r'QC',r'코드',r'지점']
for k,v in FILES.items(): print(k, v['path'], 'exists=',Path(v['path']).is_file())


plant_A ..\assets\inverterdata\inverterdata_62_-261120.csv exists= True
plant_B ..\assets\inverterdata\inverterdata_64_-261120.csv exists= True
asos ..\assets\asosdata\asosdata.csv exists= True


## 1. CSV 읽기, 전체 NULL/빈 열 제외 및 프로파일링

In [ ]:
MISSING={'','null','none','nan','n/a','na','<null>'}
TIME_NAMES=['reg_date','regdate','timestamp','datetime','date_time','measured_at','time',
            '일시','일자','측정일시','측정시간','날짜']
def read_csv_robust(path):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(path)
    for enc in ['utf-8-sig','utf-8','cp949','euc-kr']:
        for sep in [',','\t',';']:
            try:
                df=pd.read_csv(path,encoding=enc,sep=sep,dtype='string',keep_default_na=False,low_memory=False)
                if len(df.columns)<2: continue
                print(f'{path.name}: rows={len(df):,} cols={len(df.columns)} encoding={enc} sep={repr(sep)}')
                return df
            except (UnicodeDecodeError, pd.errors.ParserError, ValueError): pass
    raise ValueError(f'CSV 인코딩/구분자 확인 필요: {path}')
def normalize(raw):
    x=raw.copy()
    for c in x.columns:
        s=x[c].astype('string').str.strip()
        x[c]=s.mask(s.str.lower().isin(MISSING).fillna(False))
    return x
def timestamp_column(cols,manual):
    if manual:
        if manual not in cols: raise KeyError(f'시각 컬럼 없음: {manual}')
        return manual
    simp=lambda x:re.sub(r'[\s_\-()]','',str(x)).lower()
    matches=[c for c in cols if simp(c) in {simp(v) for v in TIME_NAMES}]
    if len(matches)!=1: raise ValueError(f'시각 컬럼 명시 필요: 후보={matches}, 전체={list(cols)}')
    return matches[0]
def parse_timestamp(s):
    try: d=pd.to_datetime(s,format='mixed',errors='coerce')
    except (TypeError,ValueError): d=pd.to_datetime(s,errors='coerce')
    if getattr(d.dt,'tz',None) is not None:
        raise ValueError('timezone-aware 날짜: KST 변환을 먼저 명시적으로 수행하세요')
    return d

dfs={}; col_rows=[]; time_cols={}
for key,spec in FILES.items():
    raw=normalize(read_csv_robust(spec['path']))
    for c in raw.columns:
        nn=int(raw[c].notna().sum())
        col_rows.append({'dataset':key,'column':c,'rows':len(raw),'valid':nn,
           'missing':len(raw)-nn,'missing_pct':100*(len(raw)-nn)/max(len(raw),1),
           'unique_nonnull':int(raw[c].nunique(dropna=True)),'all_missing':nn==0})
    tc=timestamp_column(raw.columns,spec['timestamp_col']);time_cols[key]=tc
    invalid=[c for c in raw.columns if raw[c].notna().sum()==0]
    if tc in invalid: raise ValueError(f'{key}: 시각이 모두 비어 있습니다')
    d=raw.drop(columns=invalid).copy()
    d['_source_timestamp_text']=d[tc]
    d['_timestamp']=parse_timestamp(d[tc])
    dfs[key]=d
    print(f'{key} 시각 컬럼={tc}; 전체 결측 제외 열={invalid}')
col_report=pd.DataFrame(col_rows)
col_report.to_csv(OUT/'01_column_profile.csv',index=False,encoding='utf-8-sig')

pd.set_option("display.max_rows", None)  # 전체 행 출력
pd.set_option("display.max_columns", None)  # 전체 열 출력

display(col_report)


inverterdata_62_-261120.csv: rows=227,458 cols=21 encoding=utf-8-sig sep=','
plant_A 시각 컬럼=reg_date; 전체 결측 제외 열=[]
inverterdata_64_-261120.csv: rows=213,722 cols=21 encoding=utf-8-sig sep=','
plant_B 시각 컬럼=reg_date; 전체 결측 제외 열=[]
asosdata.csv: rows=25,320 cols=20 encoding=cp949 sep=','
asos 시각 컬럼=일시; 전체 결측 제외 열=[]


,dataset,column,rows,valid,missing,missing_pct,unique_nonnull,all_missing
0,plant_A,reg_date,227458,227458,0,0.000000,227441,False
1,plant_A,sid,227458,227458,0,0.000000,1,False
2,plant_A,deviceID,227458,227458,0,0.000000,1,False
3,plant_A,multiID,227458,227458,0,0.000000,1,False
4,plant_A,errcode,227458,227458,0,0.000000,2,False
...,...,...,...,...,...,...,...,...
57,asos,전운량(10분위),25320,25315,5,0.019747,11,False
58,asos,중하층운량(10분위),25320,23073,2247,8.874408,11,False
59,asos,최저운고(100m ),25320,10458,14862,58.696682,76,False
60,asos,시정(10m),25320,25135,185,0.730648,4837,False


## 2. 약 5분 간격의 실제 수집 주기 분석

기존 15분 정각 격자 가정은 사용하지 않습니다. 설비 식별자가 설정되었다면 각 설비 내에서만 차분합니다. 중복 시각은 임의 병합하지 않고 별도 보고하며, ASOS의 누락 시간은 1시간 격자로 확인합니다.

In [ ]:
def num(s): return pd.to_numeric(s.astype('string').str.replace(',','',regex=False),errors='coerce')
def group_frame(d,key):
    idc=FEATURE_COLUMNS[key]['device_id']
    if idc and idc not in d.columns: raise KeyError(f'{key}: device_id 컬럼 없음: {idc}')
    return d.groupby(idc,dropna=False,sort=False) if idc else [('all',d)]
interval_rows=[]; time_rows=[]
for key,d in dfs.items():
    valid=d.dropna(subset=['_timestamp']).copy()
    is_asos=key=='asos'
    dup_cols=['_timestamp'] if is_asos or not FEATURE_COLUMNS[key]['device_id'] else [FEATURE_COLUMNS[key]['device_id'],'_timestamp']
    dup=valid.duplicated(subset=dup_cols,keep=False)
    valid.loc[dup].to_csv(OUT/f'02_{key}_duplicate_timestamps.csv',index=False,encoding='utf-8-sig')
    time_rows.append({'dataset':key,'rows':len(d),'invalid_ts':int(d['_timestamp'].isna().sum()),
          'duplicate_rows':int(dup.sum()),'min_time':valid['_timestamp'].min(),
          'max_time':valid['_timestamp'].max(),'unique_timestamps':int(valid['_timestamp'].nunique())})
    if is_asos:
        times=pd.DatetimeIndex(valid['_timestamp'].unique()).sort_values()
        if len(times):
            expected=pd.date_range(times.min().ceil('h'),times.max().floor('h'),freq='h')
            missing=expected.difference(times)
            pd.DataFrame({'missing_asos_hour':missing}).to_csv(OUT/'02_asos_missing_hours.csv',index=False,encoding='utf-8-sig')
    else:
        for device,part in group_frame(valid,key):
            t=part['_timestamp'].drop_duplicates().sort_values()
            gap=t.diff().dt.total_seconds().div(60).dropna()
            interval_rows.append({'dataset':key,'device_id':str(device),'interval_count':len(gap),
              'min_minutes':gap.min(),'median_minutes':gap.median(),
              'p05_minutes':gap.quantile(.05),'p95_minutes':gap.quantile(.95),
              'max_minutes':gap.max(), 'alert_gt_min_count':int((gap>GAP_ALERT_MIN).sum()),
              'near_5m_pct':100*gap.between(4,6).mean() if len(gap) else np.nan})
            pd.DataFrame({'timestamp':t.iloc[1:].values,'interval_minutes':gap.values}).to_csv(
              OUT/f'02_{key}_intervals_{re.sub(r"[^A-Za-z0-9_-]","_",str(device))[:40]}.csv',index=False,encoding='utf-8-sig')
time_summary=pd.DataFrame(time_rows);interval_summary=pd.DataFrame(interval_rows)
time_summary.to_csv(OUT/'02_time_summary.csv',index=False,encoding='utf-8-sig')
interval_summary.to_csv(OUT/'02_plant_interval_summary.csv',index=False,encoding='utf-8-sig')
display(time_summary);display(interval_summary)


## 3. 기존 파생 특징 검증 — 재생성하지 않음

설정된 열이 있을 때만 검증합니다. 정격출력·출력의 단위가 W, 누적 발전량의 단위가 설정과 일치하는지 실제 데이터 명세 확인이 필요합니다. `energy_delta_wh`의 차분은 **개별 인버터별**로 계산되었는지 검증합니다. 측정 간격과 정격출력에 따른 에너지 비율은 검증 참고값으로만 저장하며 원본 특성은 변경하지 않습니다.

In [ ]:
derived_rows=[]
for key in ['plant_A','plant_B']:
    d=dfs[key].copy();cfg=FEATURE_COLUMNS[key]
    requested={k:v for k,v in cfg.items() if k.endswith('_w') or k in ['accum_energy','energy_delta_wh','power_ratio']}
    for role,c in requested.items():
        if c and c not in d.columns: raise KeyError(f'{key} {role} 컬럼 없음: {c}')
    cpower=cfg['power_w'];crated=cfg['rated_power_w'];cratio=cfg['power_ratio']
    if cpower and crated and cratio:
        p=num(d[cpower]);rated=num(d[crated]);observed=num(d[cratio]);
        expected=p/rated.where(rated>0)
        diff=(observed-expected).abs()
        good=diff.notna()
        derived_rows.append({'dataset':key,'test':'power_ratio','tested':int(good.sum()),
          'mismatch_gt_1e-3':int((diff>1e-3).sum()),'invalid_rated':int((rated<=0).sum()),
          'ratio_gt_1':int((observed>1).sum()),'ratio_negative':int((observed<0).sum())})
    else: print(f'{key}: 출력 비율 검증 생략 (컬럼 매핑 미지정)')
    if cfg['accum_energy'] and cfg['energy_delta_wh']:
        x=d.copy();x['_acc']=num(x[cfg['accum_energy']]);x['_delta']=num(x[cfg['energy_delta_wh']])
        x=x.sort_values('_timestamp')
        idc=cfg['device_id']
        if idc:
            expected=x.groupby(idc,dropna=False)['_acc'].diff()
            minutes=x.groupby(idc,dropna=False)['_timestamp'].diff().dt.total_seconds()/60
        else:
            expected=x['_acc'].diff();minutes=x['_timestamp'].diff().dt.total_seconds()/60
        factor={'Wh':1,'kWh':1000}
        if cfg['energy_unit'] not in factor or cfg['energy_delta_unit'] not in factor:
            raise ValueError('energy_unit / energy_delta_unit은 Wh 또는 kWh로 설정')
        actual_delta_wh=x['_delta']*factor[cfg['energy_delta_unit']]
        expected_wh=expected*factor[cfg['energy_unit']]
        err=(actual_delta_wh-expected_wh).abs()
        valid=err.notna() & minutes.gt(0)
        derived_rows.append({'dataset':key,'test':'energy_delta_wh',
          'tested':int(valid.sum()),'mismatch_gt_1Wh':int((err[valid]>1).sum()),
          'negative_delta':int((actual_delta_wh<0).sum()),
          'interval_gt_alert':int((minutes>GAP_ALERT_MIN).sum())})
        if crated:
            rated=num(x[crated]); ratio=actual_delta_wh/(rated*minutes/60)
            x['_energy_ratio_check_only']=ratio.where((minutes>0)&rated.gt(0))
        x['_interval_minutes_check_only']=minutes
        x[[time_cols[key], '_timestamp','_interval_minutes_check_only','_delta']+
           (['_energy_ratio_check_only'] if '_energy_ratio_check_only' in x else [])].to_csv(
             OUT/f'03_{key}_derived_checks.csv',index=False,encoding='utf-8-sig')
    else: print(f'{key}: 누적 발전량 차분 검증 생략 (컬럼 매핑 미지정)')
derived_report=pd.DataFrame(derived_rows)
derived_report.to_csv(OUT/'03_derived_validation_summary.csv',index=False,encoding='utf-8-sig')
display(derived_report)


## 4. ASOS 시간별 결측 처리 — 변수별 제한적 선형 보간

화이트리스트에 있는 실제 연속형 항목만 처리합니다. 원본의 `NULL`/빈 값, 누락된 시간 레코드 모두 결측으로 취급합니다. **허용 구간 밖의 결측은 유지**합니다. 원본 시간별 관측이 아닌 인버터 사이 시각으로 추가 보간된 값은 별도로 `__estimated_at_plant_time` 표기합니다.

In [ ]:
asos=dfs['asos'].copy()
asos=asos.loc[asos['_timestamp'].between(START,END)].copy()
if asos['_timestamp'].isna().any(): print('ASOS 잘못된 시각 제외:',int(asos['_timestamp'].isna().sum()))
asos=asos.dropna(subset=['_timestamp']).sort_values('_timestamp')
if asos['_timestamp'].duplicated().any():
    raise ValueError('ASOS 동일 시각 중복: 02_asos_duplicate_timestamps.csv 검토 후 해결 필요')
if asos.empty: raise ValueError('분석 기간 내 ASOS 관측값이 없습니다')
if not asos['_timestamp'].eq(asos['_timestamp'].dt.floor('h')).all():
    raise ValueError('ASOS 시간별 시각이 정각이 아닙니다. 관측 시각 정의 확인 필요')
weather_cols=[]; excluded=[]
for c in asos.columns:
    if c in {time_cols['asos'],'_timestamp','_source_timestamp_text'}: continue
    if any(re.search(p,c,re.I) for p in WEATHER_BLOCK): excluded.append((c,'blocked'));continue
    if any(re.search(p,c,re.I) for p in WEATHER_ALLOW) and num(asos[c]).notna().any():
        weather_cols.append(c)
    else: excluded.append((c,'not_allowed_or_nonnumeric'))
print('보간 허용된 ASOS 컬럼:',weather_cols)
print('보간 제외된 ASOS 컬럼:',excluded[:30])
idx=pd.date_range(START,END,freq='h')
raw_hour=pd.DataFrame(index=idx)
for c in weather_cols: raw_hour[c]=pd.Series(num(asos[c]).values,index=asos['_timestamp']).reindex(idx)
def fill_internal_short(s,max_missing):
    s=s.astype('float64'); mask=s.isna()
    group=mask.ne(mask.shift(fill_value=False)).cumsum()
    lengths=mask.groupby(group).transform('sum')
    positions=np.flatnonzero(s.notna().to_numpy())
    if len(positions)<2: return s.copy(), pd.Series(False,index=s.index)
    inside=pd.Series((np.arange(len(s))>positions[0])&(np.arange(len(s))<positions[-1]),index=s.index)
    fillmask=mask & inside & lengths.le(max_missing)
    interpolated=s.interpolate(method='time',limit_area='inside')
    result=s.where(~fillmask,interpolated)
    return result, fillmask & result.notna()
hourly=raw_hour.copy();fill_rows=[]
for c in weather_cols:
    hourly[c],flag=fill_internal_short(raw_hour[c],MAX_MISSING_ASOS_HOURS)
    hourly[c+'__filled_hourly']=flag
    fill_rows.append({'column':c,'missing_before':int(raw_hour[c].isna().sum()),
         'hourly_filled':int(flag.sum()),'missing_after':int(hourly[c].isna().sum())})
    if '습도' in c:
        bad=hourly[c].notna()&~hourly[c].between(0,100)
        if bad.any():warnings.warn(f'{c}: 습도 0~100 범위 밖 {bad.sum()}개 관측. 검토 필요')
hourly.index.name='asos_hour';hourly.to_csv(OUT/'04_asos_hourly_clean.csv',encoding='utf-8-sig')
fill_report=pd.DataFrame(fill_rows)
fill_report.to_csv(OUT/'04_asos_fill_summary.csv',index=False,encoding='utf-8-sig')
display(fill_report)


## 5. 인버터 실제 시각에 ASOS 연속형 값 매칭

**각 인버터 행을 유지**하고 해당 행의 측정 시각을 기준으로 ASOS 정각의 이전·이후 유효 관측값이 모두 있을 때만 선형 보간합니다. 장시간 결측을 건너뛰지 않도록 두 관측값 사이 최대 3시간 제약을 둡니다. **기존 시간별 내부 보간으로 채운 시각은 허용된 범위 내**이므로 사용 가능합니다. 강수량·일사량 등 시간 집계형 변수는 이 단계에서 자동 결합하지 않습니다.

오프라인 실험이므로 미래 ASOS 관측값을 사용할 수 있으며, 실시간 추론에는 그대로 이식할 수 없습니다. 복수 발전소의 시각을 일치시키지 않습니다.

In [ ]:
def weather_on_plant_time(plant, hourly, columns):
    out=plant.copy(); ts=out['_timestamp']
    good=ts.notna() & ts.between(START,END)
    hprev=ts.dt.floor('h'); hnext=hprev+pd.Timedelta(hours=1)
    for c in columns:
        s=hourly[c]
        # 이전/이후 정각의 보간 후 값. 해당 시간의 관측값이 있으면 정확히 사용.
        left=pd.Series(hprev.map(s),index=out.index,dtype='float64')
        right=pd.Series(hnext.map(s),index=out.index,dtype='float64')
        fraction=(ts-hprev).dt.total_seconds()/3600
        valid=good & left.notna() & (fraction.eq(0) | right.notna())
        # 시각별 보간된 ASOS 값은 원본 관측 사이의 gap이 3h 이하인 경우에만 존재.
        joined=left+(right-left)*fraction
        joined=joined.where(valid)
        joined=joined.mask(valid & fraction.eq(0),left)
        # 여기서는 미보간 ASOS 시간의 바로 앞/뒤 관측의 시각 간격은 1h (시간별 재색인)
        out['asos__'+c]=joined
        out['asos__'+c+'__estimated_at_plant_time']=(valid & fraction.ne(0)).astype('boolean')
        leftflag=pd.Series(hprev.map(hourly[c+'__filled_hourly']),index=out.index).fillna(False).astype(bool)
        rightflag=pd.Series(hnext.map(hourly[c+'__filled_hourly']),index=out.index).fillna(False).astype(bool)
        out['asos__'+c+'__uses_hourly_imputation']=(valid & (leftflag | (rightflag & fraction.ne(0)))).astype('boolean')
    return out

merged={}; join_rows=[]
for key in ['plant_A','plant_B']:
    d=dfs[key]
    merged[key]=weather_on_plant_time(d,hourly,weather_cols)
    merged[key].to_csv(OUT/f'05_{key}_asos_aligned_original_timestamps.csv',index=False,encoding='utf-8-sig')
    for c in weather_cols:
        s=merged[key]['asos__'+c]
        join_rows.append({'dataset':key,'weather_column':c,'plant_rows':len(d),
          'matched':int(s.notna().sum()),'unmatched':int(s.isna().sum()),
          'matched_pct':round(100*s.notna().mean(),2) if len(s) else np.nan,
          'interpolated_at_plant_time':int(merged[key]['asos__'+c+'__estimated_at_plant_time'].sum()),
          'uses_hourly_imputation':int(merged[key]['asos__'+c+'__uses_hourly_imputation'].sum())})
join_report=pd.DataFrame(join_rows)
join_report.to_csv(OUT/'05_asos_join_summary.csv',index=False,encoding='utf-8-sig')
display(join_report)


## 6. 결과 검증 및 범위

출력 폴더에서 `01_column_profile.csv`, `02_plant_interval_summary.csv`, `03_derived_validation_summary.csv`, `04_asos_fill_summary.csv`, `05_asos_join_summary.csv`를 우선 검토하십시오. 각 발전소의 `05_*_asos_aligned_original_timestamps.csv`는 **원본 수집 시각 그대로** 기상 변수와 결합된 데이터입니다.

- CSV에 별도 자료형·단위·식별자가 확인되기 전까지 무리한 자동 컬럼 매핑을 하지 않습니다.
- 시간별 누적 관측값(일사·강수·일조)은 관측시간 정의가 정해질 때까지 제외합니다.
- 10개 가상 발전소 생성, E0/E1 Isolation Forest 학습 및 OPC UA 구독은 후속 단계에서 수행합니다.
- 실제 CSV가 아닌 테스트 데이터 실행 결과는 실제 발전소의 결측률이나 수집 주기 추정치로 해석하지 않습니다.
